# Tutorial 1: Time-Series Forecasting with PatchTST

In this notebook, we demonstrate how to use the `PatchTST` model (SOTA in long-term forecasting as of 2023) within the Industrial-Time-Series-AI framework to forecast multivarite sensor readings.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
from datasets.dataloader import get_dummy_swat_dataloader
from models.patchtst import PatchTST
from forecasting.pipeline import ForecastingPipeline
from visualization.plot_utils import plot_multivariate_timeseries

## 1. Load Data

We use the built-in SWaT dummy dataloader. In a real scenario, this would load the actual SWaT dataset.

In [ ]:
# Forecasting Task: Given 96 time steps, predict the next 24
train_loader = get_dummy_swat_dataloader(batch_size=32, window_size=120, num_samples=2000)
val_loader = get_dummy_swat_dataloader(batch_size=32, window_size=120, num_samples=500)

## 2. Initialize PatchTST

PatchTST operates independently on channels and uses patching to extract local semantic context.

In [ ]:
model = PatchTST(
    num_features=51, 
    seq_len=96, 
    pred_len=24, 
    patch_len=16, 
    stride=8,
    d_model=64,
    nhead=4,
    num_layers=2
)

print(f"Total Model Parameters: {sum(p.numel() for p in model.parameters())}")

## 3. Train Model

Using the provided generic `ForecastingPipeline`.

In [ ]:
pipeline = ForecastingPipeline(model, learning_rate=1e-3)

print("Note: In this tutorial we format the windows in the pipeline.")
# Mock adaptation for the pipeline: the dataloader yields (x) in the dummy, we need (x, y)
# In a real setup, `create_forecasting_windows` from feature_engineering handles this.